# LTV Prediction: Feature Engineering

In this notebook, we will prepare the dataset for our Lifetime Value (LTV) prediction model. The process involves:
1.  Defining our target variable: total spend in the 90 days after a customer's first purchase.
2.  Creating features based on a customer's *first* order (e.g., initial payment value, review score for the first order).
3.  Combining these into a final modeling dataset.

In [1]:
import pandas as pd
import numpy as np
import sys
import os

# Add the project's root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Load the processed master dataset we created with main.py
processed_data_path = '../data/processed/master_dataset.csv'
master_df = pd.read_csv(processed_data_path)

# Convert key date columns to datetime objects
date_cols = ['order_purchase_timestamp', 'order_approved_at']
for col in date_cols:
    master_df[col] = pd.to_datetime(master_df[col], errors='coerce')

# Drop rows where essential data is missing
master_df.dropna(subset=['order_purchase_timestamp', 'customer_unique_id', 'payment_value'], inplace=True)

### Creating the Target Variable and Features

This is the most critical data manipulation step for our model. For each unique customer, we need to find their first purchase and then calculate their total spend in the subsequent 90-day window. The features for the model will be based *only* on that first transaction.

In [2]:
# Find the first purchase for each customer
first_purchase_df = master_df.loc[master_df.groupby('customer_unique_id')['order_purchase_timestamp'].idxmin()]

# --- Create the Target Variable (LTV_90_days) ---
# For each customer, calculate their total spend in the 90 days AFTER their first purchase.
target_df = master_df.groupby('customer_unique_id').apply(
    lambda x: x[
        (x['order_purchase_timestamp'] > x['order_purchase_timestamp'].min()) &
        (x['order_purchase_timestamp'] <= x['order_purchase_timestamp'].min() + pd.Timedelta(days=90))
    ]['payment_value'].sum()
).reset_index(name='ltv_90_days')


# --- Create Features from the First Purchase ---
# Our features can only be things we know at the time of the first purchase.
feature_df = first_purchase_df[[
    'customer_unique_id',
    'payment_value',          # Value of the first purchase
    'payment_installments',   # Number of installments for the first purchase
    'review_score',           # Review score for the first order
    'freight_value',          # Shipping cost of the first order
    'product_category',       # Category of the first item purchased
    'product_photos_qty',
    'product_description_lenght'
]].copy()


# --- Combine into the Final Modeling Dataset ---
modeling_df = pd.merge(feature_df, target_df, on='customer_unique_id')

# Handle missing values (e.g., fill with median or mean for numerical, 'unknown' for categorical)
modeling_df['review_score'].fillna(modeling_df['review_score'].median(), inplace=True)
modeling_df['product_photos_qty'].fillna(modeling_df['product_photos_qty'].median(), inplace=True)
modeling_df['product_description_lenght'].fillna(modeling_df['product_description_lenght'].median(), inplace=True)
modeling_df['product_category'].fillna('unknown', inplace=True)

# Convert categorical feature into numerical using one-hot encoding
modeling_df = pd.get_dummies(modeling_df, columns=['product_category'], drop_first=True)

print("Modeling dataset created successfully.")
modeling_df.head()

Modeling dataset created successfully.


C:\Users\shaik\AppData\Local\Temp\ipykernel_28500\3417438040.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  target_df = master_df.groupby('customer_unique_id').apply(
C:\Users\shaik\AppData\Local\Temp\ipykernel_28500\3417438040.py:32: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the op

,customer_unique_id,payment_value,payment_installments,review_score,freight_value,product_photos_qty,product_description_lenght,ltv_90_days,product_category_air_conditioning,product_category_art,...,product_category_signaling_and_security,product_category_small_appliances,product_category_small_appliances_home_oven_and_coffee,product_category_sports_leisure,product_category_stationery,product_category_tablets_printing_image,product_category_telephony,product_category_toys,product_category_unknown,product_category_watches_gifts
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90,8.0,5.0,12.00,1.0,236.0,0.0,False,False,...,False,False,False,False,False,False,False,False,False,False
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19,1.0,4.0,8.29,1.0,635.0,0.0,False,False,...,False,False,False,False,False,False,False,False,False,False
2,0000f46a3911fa3c0805444483337064,86.22,8.0,3.0,17.22,3.0,177.0,0.0,False,False,...,False,False,False,False,True,False,False,False,False,False
3,0000f6ccb0745a6a4b88665a16c9f078,43.62,4.0,4.0,17.63,5.0,1741.0,0.0,False,False,...,False,False,False,False,False,False,True,False,False,False
4,0004aac84e0df4da2b147fca70cf8255,196.89,6.0,5.0,16.89,3.0,794.0,0.0,False,False,...,False,False,False,False,False,False,True,False,False,False
